In [23]:
import scripts

smiles = "CCC(O[O])C1CO1"
reactant = scripts.mol_from_smiles(smiles)

PROTON_TRANSFER = "([Cv3,Ov1:1].[Cv4,Ov2:2]([H:3]))>>([Cv4,Ov2:1]([H:3]).[Cv3,Ov1:2])"
product_sets = scripts.reaction(reactant, PROTON_TRANSFER)
scripts.process_rdkit_reaction(reactant, product_sets)

# RING_OPENING = "[Cv4,Ov2:1]-@[Cv4,Ov2:2]-[Cv3,Ov1:3]>>([Cv3,Ov1:1].[Cv4,Ov2:2]=[Cv4,Ov2:3])"
# product_sets = scripts.reaction(reactant, RING_OPENING)
# scripts.process_rdkit_reaction(reactant, product_sets)

In [8]:
from pathlib import Path
import py3Dmol

def plot_model(file: str, label: bool = True):
    xyz_block = Path(file).read_text()
    cleaned_xyz_block = xyz_block.replace(">\n", "")

    first_line = cleaned_xyz_block.splitlines()[0]
    atom_count = int(first_line.strip())

    viewer = py3Dmol.view()
    viewer.addModelsAsFrames(cleaned_xyz_block, "xyz")

    viewer.setStyle({"stick": {}, "sphere": {"scale": 0.25}})

    if label:
        for idx in range(atom_count):
            viewer.addLabel(
                idx,
                {
                    "backgroundOpacity": 0.0,
                    "fontColor": "black",
                    "alignment": "center",
                    "inFront": True,
                },
                {"index": idx},
            )

    viewer.zoomTo()
    viewer.animate({"loop": "backAndForth"})
    viewer.show()

In [17]:
plot_model(file = "/home/tns97255/test/neb/product.xyz")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [9]:
plot_model(file = "/home/tns97255/test/results/T85-0/freq.hess.v006.xyz")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [2]:
import os
import re

def analyze_range(start, end, base_path="results"):
    """
    Analyze T# folders from start to end (inclusive).
    
    Extracts:
        - Total Energy
        - Imaginary mode numbers and frequencies
    
    Returns:
        dict containing all results and lowest-energy structure
    """
    
    results = {}
    
    energy_pattern = re.compile(r"Total Energy:\s*([-+]?\d*\.\d+|\d+)")
    
    # Capture BOTH mode number and frequency
    imag_pattern = re.compile(
        r"Imaginary Frequency:\s*(\d+):\s*([-+]?\d*\.\d+|\d+)\s*cm\*\*-1.*imaginary mode",
        re.IGNORECASE
    )
    
    for i in range(start, end + 1):
        folder_name = f"T{i}"
        file_path = os.path.join(base_path, folder_name, "result.txt")
        
        if not os.path.exists(file_path):
            print(f"Warning: {file_path} not found.")
            continue
        
        with open(file_path, "r") as f:
            content = f.read()
        
        energy_match = energy_pattern.search(content)
        imag_matches = imag_pattern.findall(content)
        
        if not energy_match:
            print(f"Warning: No energy found in {folder_name}")
            continue
        
        energy = float(energy_match.group(1))
        
        # Store mode number + frequency together
        imaginary_modes = [
            {"mode_number": int(mode), "frequency": float(freq)}
            for mode, freq in imag_matches
        ]
        
        results[i] = {
            "energy": energy,
            "imaginary_modes": imaginary_modes
        }
    
    if not results:
        print("No valid results found.")
        return None
    
    # Determine lowest energy
    lowest_key = min(results, key=lambda k: results[k]["energy"])
    lowest_energy = results[lowest_key]["energy"]
    lowest_imag_modes = results[lowest_key]["imaginary_modes"]
    
    print("===== Summary =====")
    print(f"Checked folders: T{start} to T{end}")
    print(f"Lowest energy: T{lowest_key}")
    print(f"Energy: {lowest_energy} kcal/mol")
    
    if lowest_imag_modes:
        print("Imaginary mode(s):")
        for mode in lowest_imag_modes:
            print(f"  Mode {mode['mode_number']}: {mode['frequency']} cm**-1")
    else:
        print("No imaginary modes found.")
    
    return {
        "all_results": results,
        "lowest_energy_folder": lowest_key,
        "lowest_energy": lowest_energy,
        "imaginary_modes": lowest_imag_modes
    }


# Example:
# analyze_range(1, 4)


In [23]:
analyze_range(150,157)

===== Summary =====
Checked folders: T150 to T157
Lowest energy: T151
Energy: -263950.2579 kcal/mol
Imaginary mode(s):
  Mode 6: -1028.71 cm**-1
  Mode 7: -23.95 cm**-1


{'all_results': {150: {'energy': -263949.8117,
   'imaginary_modes': [{'mode_number': 6, 'frequency': -1016.9}]},
  151: {'energy': -263950.2579,
   'imaginary_modes': [{'mode_number': 6, 'frequency': -1028.71},
    {'mode_number': 7, 'frequency': -23.95}]},
  152: {'energy': -263949.7017,
   'imaginary_modes': [{'mode_number': 6, 'frequency': -1018.52},
    {'mode_number': 7, 'frequency': -22.61}]},
  154: {'energy': -263949.8848,
   'imaginary_modes': [{'mode_number': 6, 'frequency': -1015.24}]},
  155: {'energy': -263950.1413,
   'imaginary_modes': [{'mode_number': 6, 'frequency': -1028.35}]},
  156: {'energy': -263949.6226,
   'imaginary_modes': [{'mode_number': 6, 'frequency': -1017.14}]},
  157: {'energy': -263949.8653,
   'imaginary_modes': [{'mode_number': 6, 'frequency': -1029.74}]}},
 'lowest_energy_folder': 151,
 'lowest_energy': -263950.2579,
 'imaginary_modes': [{'mode_number': 6, 'frequency': -1028.71},
  {'mode_number': 7, 'frequency': -23.95}]}

In [ ]:
from scripts import reaction_graphs, transition_graph
import automol
from inputs import SCAN_INP, TRANS_FREQ_INP, CALC_INP, TRANSITION_SH
from pathlib import Path

reactant_amchi = "AMChI=1/C5H9O/c1-3-4-5-2-6-5/h2,5H,3-4H2,1H3"
product_amchis = ["AMChI=1/C5H9O/c1-4-5-2-3-6/h2-3H,4-5H2,1H3"]
path = Path("results/T125")

for i, reaction in enumerate(reaction_graphs(reactant_amchi, tuple(product_amchis))):
    transition_gra = transition_graph(reaction)
    try:
        transition_amchi = automol.graph.amchi(transition_gra)

        transition_geo = automol.graph.geometry(transition_gra)
        transition_xyz = automol.geom.xyz_string(transition_geo)

        formed = automol.graph.ts.forming_bond_keys(transition_gra)
        broken = automol.graph.ts.breaking_bond_keys(transition_gra)

        dmat_angstrom = (
            automol.geom.distance_matrix(transition_geo) * 0.529177
        )

        for broken_bond in broken:
            a, b = broken_bond
            if len(formed) > 0:
                for formed_bond in formed:
                    shared = broken_bond & formed_bond
                    if len(shared) == 1:
                        c = next(iter(formed_bond - shared))
                        dist = dmat_angstrom[b, c]
                        active_atoms = f"{b} {c}"
                        scan = (
                            f"scan B {active_atoms} = {dist:.3f}, 0.7, 100"
                        )

            else:
                active_atoms = f"{a} {b}"
                scan = f"scan B {active_atoms} = {dmat_angstrom[a, b]:.3f}, 2.5, 100"

    except Exception:
        continue
    
    out = Path(f"{path}-{i}")
    out.mkdir(exist_ok=True)
    (out / "guess.xyz").write_text(transition_xyz)
    (out / "scan.inp").write_text(
        SCAN_INP.replace("[SCAN]", scan).replace("[ATOMS]", active_atoms)
    )
    (out / "freq.inp").write_text(
        TRANS_FREQ_INP.replace("[ATOMS]", active_atoms)
    )
    (out / "calc.inp").write_text(CALC_INP)
    (out / "submit.sh").write_text(TRANSITION_SH)

NotImplementedError: 